In [ ]:
# 修改后的代码一：成本敏感性分析版本 (替换优化为你的模拟)
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
pd.set_option('display.float_format', '{:.6f}'.format)

# Gamma parameters
gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 0. Read data (使用你的原始逻辑，只取cr>0的部分)
# =============================================================================
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv')
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

# =============================================================================
# 参数 (C_MANT, C_COPER 固定, C_leak 变化)
# =============================================================================
T, base_year, Y = 7, 2024, 365
mu_coper, mu_mant = 1.6, 4.2
C_MANT = 48000
C_COPER = 15600
nb_coper, nb_mant = 3, 14

# =============================================================================
# 1. 2023 baseline
# =============================================================================
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]
print("2023 baseline (所有csemCur):")
print(df_alpha_2023.round(4))

# =============================================================================
# 2. 模拟函数 (你的原始模拟逻辑，返回详细结果)
# =============================================================================
def run_simulation(C_leak):
    # Initialize states (你的原始逻辑)
    csem_states = {}
    for i, row in df_alpha_2023.iterrows():
        csemCur = int(row["csemCur"])
        alpha, delta_p = row["alpha_2023"], row["Pression Nominale Relative"] - row["CR_threshold"]
        cr_init = row["cr"] if row["cr"] > 0 else 0
        
        csem_states[csemCur] = {
            'cr_current': cr_init,
            'age_2023_fixed': Age_2023_fixed[i],
            'current_age': Age_2023_fixed[i] + 1,  # 2024开始
            'alpha': alpha, 'pression_nominale': row["Pression Nominale Relative"],
            'cr_threshold': row["CR_threshold"], 'volume_csem': row["Volume"],
            'csem_K': 146 * row["Volume"] / (8.314 * 293.15 * 10**3)
        }

    # Simulation loop (你的原始逻辑)
    rows = []
    for t in range(T):
        year = base_year + t
        year_offset = year - 2023
        
        # Find triggered & assign tasks (你的逻辑)
        triggered_csem = []
        for csemCur, state in csem_states.items():
            cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
            leak = cr * delta_p * (10**5) * state['csem_K']
            interval = 365 / cr if cr > 0 else np.inf
            if leak >= 5 or interval <= 60:
                triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'state': state})
        
        triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
        task_assignments = {}
        for i, info in enumerate(triggered_csem):
            csemCur = info['csemCur']
            if i < nb_mant: task_assignments[csemCur] = 'mant'
            elif i < nb_mant + nb_coper: task_assignments[csemCur] = 'coper'
        
        # Process all
        for csemCur, state in csem_states.items():
            cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
            leak = cr * delta_p * (10**5) * state['csem_K']
            interval = 365 / cr if cr > 0 else np.inf
            triggered = (leak >= 5) or (interval <= 60)
            
            leak_cost = leak * C_leak  # 使用传入的C_leak
            task_type = task_assignments.get(csemCur)
            action_cost = { 'mant': C_MANT, 'coper': C_COPER }.get(task_type, 0)
            
            rows.append(dict(csemCur=csemCur, year=year, cr=cr, leak=leak, interval=interval,
                            triggered=triggered, task_type=task_type,
                            leak_cost=leak_cost, action_cost=action_cost, total_cost=leak_cost+action_cost))
            
            # Update state (你的逻辑)
            next_offset = year_offset + 1
            next_age = state['age_2023_fixed'] + next_offset
            
            if task_type == 'mant':
                state['cr_current'] /= mu_mant
            elif task_type == 'coper':
                state['cr_current'] /= mu_coper
            else:
                state['cr_current'] += compute_gamma_h(next_age, gamma_alpha, gamma_beta)
            state['current_age'] = next_age

    sim_df = pd.DataFrame(rows)
    sim_df['csemCur'] = sim_df['csemCur'].astype(int)
    
    # Summary (你的原始汇总逻辑)
    cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
    cost_summary.columns = ["csemCur", "total_T years_cost"]
    leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
    leak_summary.columns = ["csemCur", "total_leak_kg"]
    
    total_leak_kg = sim_df['leak'].sum().round(3)
    total_cost = sim_df['total_cost'].sum().round(0)
    total_leak_cost = sim_df['leak_cost'].sum().round(0)
    total_action_cost = sim_df['action_cost'].sum().round(0)
    total_coper_actions = (sim_df['task_type'] == 'coper').sum()
    total_mant_actions = (sim_df['task_type'] == 'mant').sum()
    
    print(f"\n✅ C_leak = {C_leak:,.0f} | Total Cost: {total_cost:,.0f}")
    print(f"Total leak: {total_leak_kg:.1f} kg | Action cost: {total_action_cost:,.0f} | Leak cost: {total_leak_cost:,.0f}")
    print(f"Total COPER: {total_coper_actions} | Total MANT: {total_mant_actions}")
    
    return {
        "C_leak": C_leak,
        "total_cost": total_cost,
        "total_leak_kg": total_leak_kg,
        "leak_cost": total_leak_cost,
        "action_cost": total_action_cost,
        "total_coper_actions": total_coper_actions,
        "total_mant_actions": total_mant_actions,
        "status": "simulation_complete"
    }, sim_df

# =============================================================================
# 3. 成本敏感性分析 (模仿优化代码的格式)
# =============================================================================
base_C_leak = 25200 * 0.100  # 基准值
C_leak_list = [
    base_C_leak * 0.01,
    base_C_leak * 0.1,
    base_C_leak * 0.5,
    base_C_leak * 1,
    base_C_leak * 1.5,
    base_C_leak * 2.0,
    base_C_leak * 3.0,
    base_C_leak * 4.0,
    base_C_leak * 5.0,
    base_C_leak * 10.0,
    base_C_leak * 100.0,
    base_C_leak * 1000.0,
    base_C_leak * 10000.0,
    base_C_leak * 100000.0
]

results = []
detailed_dfs = {}  # 保存详细轨迹

for C_leak in C_leak_list:
    res, sim_df = run_simulation(C_leak)
    results.append(res)
    detailed_dfs[C_leak] = sim_df  # 可选：保存详细数据

# Summary table
result_df = pd.DataFrame(results)
print("\n========== Summary of Sensitivity Analysis (Simulation) ==========")
print(result_df.round(0))

# 保存
filename = "simulation_sensitivity_C_leak"
result_df.to_csv(f"{filename}_summary.csv", index=False)
print(f"\n✅ Saved {filename}_summary.csv")

# 统计输出 (模仿优化代码)
print('\nNumber of COPER actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_coper_actions']:.0f} COPER actions") 

print('\nNumber of MANT actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_mant_actions']:.0f} MANT actions")

2023 baseline (所有csemCur):
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold  \
0    5000252447    0.000300                    3.500000      3.199000   
1    5000252448    0.000500                    3.500000      3.199000   
2    5000252449    0.000100                    3.500000      3.199000   
3    5000252450    0.000900                    3.500000      3.199000   
4    5000252524    0.000400                    3.500000      3.199000   
..          ...         ...                         ...           ...   
219  5001342228    0.001000                    3.500000      3.199000   
220  5001342229    0.000300                    3.500000      3.199000   
221  5002343705    0.000300                    3.500000      3.199000   
222  5002343706    0.000100                    3.500000      3.199000   
223  5002343707    0.000200                    3.500000      3.199000   

      Volume       cr  
0   6.317600 0.000000  
1   6.317600 0.000000  
2   6.302500 0.000000  


In [ ]:
# 纯模拟cr数量的变化不安排维修任务

import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc

pd.set_option("display.float_format", "{:,.4f}".format)

data_path = '/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv'
new_df = pd.read_csv(data_path)
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

gamma_alpha = 5.682011911867158
gamma_beta = 0.25018840842979584

def compute_gamma_h(age):
    z = gamma_beta * age
    if z <= 0: return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 初始化：每个csemCur自己的年龄序列
# =============================================================================
csem_states = {}
df_init = new_df.groupby("csemCur")[["cr"]].last().reset_index()
csem_list = sorted(df_init['csemCur'].tolist())

for i, csemCur in enumerate(csem_list):
    age_2023 = Age_2023_fixed[i]
    csem_states[csemCur] = {
        'cr_2023': max(df_init[df_init['csemCur']==csemCur]['cr'].iloc[0], 0.0),
        'age_2023': age_2023,
        'cr_current': 0.0
    }

print(f"✅ {len(csem_states)} 个csemCur，每个用自己的年龄序列")

# =============================================================================
# 模拟：每个csemCur用自己的年龄算h值
# =============================================================================
T = 17
rows_cr = []
rows_h = []

print("\n🎯 每个csemCur → 每年自己的h(age_t)值:")
print("="*100)

for t in range(T):
    sim_year = 2024 + t
    print(f"\nYear {sim_year}:")
    
    for csemCur, state in csem_states.items():
        # ✅ 每个csemCur用自己的当年年龄算h值
        age_this_year = state['age_2023'] + t  # 2024用age_2023, 2025用age_2023+1...
        h_this_year = compute_gamma_h(age_this_year)
        
        # 更新CR
        if t == 0:
            state['cr_current'] = state['cr_2023'] + h_this_year
        else:
            state['cr_current'] += h_this_year
        
        # 记录
        rows_cr.append({
            'csemCur': csemCur,
            'year': sim_year,
            'CR': state['cr_current'],
            'age_this_year': age_this_year
        })
        
        rows_h.append({
            'csemCur': csemCur,
            'year': sim_year,
            'h_value': h_this_year,
            'age_used': age_this_year  # 自己的年龄！
        })
        
        # 打印前10个示例
        if csemCur <= list(csem_states.keys())[9]:
            print(f"  csemCur {csemCur}: age={age_this_year:.1f}, h={h_this_year:.4f}, CR={state['cr_current']:.4f}")

sim_df_cr = pd.DataFrame(rows_cr)
sim_df_h = pd.DataFrame(rows_h)

# =============================================================================
# 保存CSV
# =============================================================================
pivot_cr = sim_df_cr.pivot(index='csemCur', columns='year', values='CR').round(4)
pivot_cr.to_csv('1_CR_透视表.csv')

pivot_h = sim_df_h.pivot(index='csemCur', columns='year', values='h_value').round(4)
pivot_h.to_csv('2_h值_每个csemCur_每年.csv')  # ✅ 每个csemCur自己的h值！

print("\n✅ 保存完成:")
print("2_h值_每个csemCur_每年.csv ← **你要的精确h值**")
print("\n📊 h值示例（前5 csemCur，前3年）:")
print(pivot_h.head()[2024:2027].round(4))

✅ 224 个csemCur，每个用自己的年龄序列

🎯 每个csemCur → 每年自己的h(age_t)值:

Year 2024:
  csemCur 5000252447: age=40.0, h=0.1477, CR=0.1477
  csemCur 5000252448: age=40.0, h=0.1477, CR=0.1477
  csemCur 5000252449: age=40.0, h=0.1477, CR=0.1477
  csemCur 5000252450: age=40.0, h=0.1477, CR=0.1477
  csemCur 5000252524: age=40.0, h=0.1477, CR=0.1477
  csemCur 5000252528: age=41.0, h=0.1498, CR=0.1498
  csemCur 5000252529: age=41.0, h=0.1498, CR=0.1498
  csemCur 5000252531: age=39.3, h=0.1461, CR=0.1461
  csemCur 5000252544: age=41.0, h=0.1498, CR=0.1498
  csemCur 5000252545: age=39.3, h=0.1461, CR=0.1461

Year 2025:
  csemCur 5000252447: age=41.0, h=0.1498, CR=0.2975
  csemCur 5000252448: age=41.0, h=0.1498, CR=0.2975
  csemCur 5000252449: age=41.0, h=0.1498, CR=0.2975
  csemCur 5000252450: age=41.0, h=0.1498, CR=0.2975
  csemCur 5000252524: age=41.0, h=0.1498, CR=0.2975
  csemCur 5000252528: age=42.0, h=0.1518, CR=0.3016
  csemCur 5000252529: age=42.0, h=0.1518, CR=0.3016
  csemCur 5000252531: age=40.3, h=0

In [29]:
# 纯模拟由模拟alpha计算而来的cr数量的变化不安排维修任务

import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc

# pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option('display.precision', 8)  # 显示最多8位小数
pd.set_option('display.float_format', lambda x: '%.8f' % x)  # 强制保留8位小数，避免科学计数法

# 你的数据路径（需包含 'csemCur', 'alpha_2023'(原cr列), 'Age 2023', 'pression_nominale', 'cr_threshold'）
data_path = '/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv'
new_df = pd.read_csv(data_path)
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

gamma_alpha = 5.682011911867158
gamma_beta = 0.25018840842979584

def compute_gamma_h(age):
    z = gamma_beta * age
    if z <= 0: return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 计算每个csemCur的deltaP = pression_nominale - CR_threshold
# =============================================================================
df_init = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold"]].last().reset_index()
csem_list = sorted(df_init['csemCur'].tolist())

deltaP_dict = {}
for _, row in df_init.iterrows():
    csemCur = row['csemCur']
    deltaP = row['Pression Nominale Relative'] - row['CR_threshold']
    # deltaP_dict[csemCur] = max(deltaP, 0.001)  # 避免除零
    deltaP_dict[csemCur] = deltaP  

Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

# 初始化状态
csem_states = {}
for i, csemCur in enumerate(csem_list):
    alpha_2023 = df_init[df_init['csemCur']==csemCur]['alpha'].iloc[0]
    age_2023 = Age_2023_fixed[i]
    h_2023 = compute_gamma_h(age_2023)
    
    csem_states[csemCur] = {
        # 'alpha_2023': max(alpha_2023, 0.0),
        'alpha_2023': alpha_2023,
        'age_2023': age_2023,
        'h_2023': h_2023,
        'deltaP': deltaP_dict[csemCur]
    }

print(f"✅ {len(csem_states)} 个csemCur，每个有自己的alpha_2023、年龄、deltaP")
print(f"deltaP范围: {min(deltaP_dict.values()):.4f} ~ {max(deltaP_dict.values()):.4f}")

# =============================================================================
# 模拟过程
# =============================================================================
T = 17
rows = []

print("\n🎯 每个csemCur → 每年alpha和CR (用自己的deltaP):")
print("="*100)

for t in range(T):
    sim_year = 2024 + t
    if t < 3:  # 只打印前3年示例
        print(f"\nYear {sim_year}:")
    
    for csemCur, state in csem_states.items():
        age_this_year = state['age_2023'] + t + 1 
        h_this_year = compute_gamma_h(age_this_year)
        
        alpha_this_year = state['alpha_2023'] / state['h_2023'] * h_this_year
        cr_this_year = 365 * alpha_this_year / state['deltaP']  # 用csemCur自己的deltaP
        # cr_this_year = alpha_this_year * 10**(-5) / state['deltaP']  # 用csemCur自己的deltaP
        
        rows.append({
            'csemCur': csemCur,
            'year': sim_year,
            'alpha': alpha_this_year,
            'CR': cr_this_year,
            'h_value': h_this_year,
            'age_this_year': age_this_year,
            'deltaP': state['deltaP']
        })
        
        # 示例打印（前10个csem，前3年）
        if t < 3 and csemCur <= list(csem_states.keys())[9]:
            print(f"  csemCur {csemCur}: age={age_this_year:.1f}, h={h_this_year:.8f}, alpha={alpha_this_year:.8f}, deltaP={state['deltaP']:.8f}, CR={cr_this_year:.8f}")

sim_df = pd.DataFrame(rows)

# =============================================================================
# 保存结果
# =============================================================================
pivot_alpha = sim_df.pivot(index='csemCur', columns='year', values='alpha').round(4)
pivot_alpha.to_csv('1_alpha_每个csemCur_每年.csv')

pivot_cr = sim_df.pivot(index='csemCur', columns='year', values='CR').round(4)
pivot_cr.to_csv('2_CR_每个csemCur_每年.csv')

# 完整结果（含deltaP）
sim_df.to_csv('3_完整结果_含deltaP.csv', index=False)

print("\n✅ 保存完成:")
print("- 1_alpha_每个csemCur_每年.csv")
print("- 2_CR_每个csemCur_每年.csv")
print("- 3_完整结果_含deltaP.csv （所有列，包括每个csem的deltaP）")

print("\n📊 示例（前5 csemCur，前3年）:")
print("Alpha:")
print(pivot_alpha.head()[2024:2027].round(4))
print("\nCR:")
print(pivot_cr.head()[2024:2027].round(4))

✅ 224 个csemCur，每个有自己的alpha_2023、年龄、deltaP
deltaP范围: 0.2838 ~ 0.3010

🎯 每个csemCur → 每年alpha和CR (用自己的deltaP):

Year 2024:
  csemCur 5000252447: age=41.0, h=0.14981114, alpha=0.00029102, deltaP=0.30100000, CR=0.35289707
  csemCur 5000252448: age=41.0, h=0.14981114, alpha=0.00050478, deltaP=0.30100000, CR=0.61210326
  csemCur 5000252449: age=41.0, h=0.14981114, alpha=0.00009676, deltaP=0.30100000, CR=0.11733345
  csemCur 5000252450: age=41.0, h=0.14981114, alpha=0.00088785, deltaP=0.30100000, CR=1.07663170
  csemCur 5000252524: age=41.0, h=0.14981114, alpha=0.00035973, deltaP=0.30100000, CR=0.43622324
  csemCur 5000252528: age=42.0, h=0.15183175, alpha=0.00023059, deltaP=0.30100000, CR=0.27961975
  csemCur 5000252529: age=42.0, h=0.15183175, alpha=0.00014061, deltaP=0.30100000, CR=0.17051167
  csemCur 5000252531: age=40.3, h=0.14824205, alpha=0.00077913, deltaP=0.30100000, CR=0.94479469
  csemCur 5000252544: age=42.0, h=0.15183175, alpha=0.00029530, deltaP=0.30100000, CR=0.35809223
  csemC

In [19]:
import pandas as pd

data_path = '/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv'
df = pd.read_csv(data_path)

print("="*60)
print("🔍 alpha列零值检测报告 (每个csemCur一行)")
print("="*60)
print(f"总csemCur数量: {len(df):,}")

alpha_values = df['alpha']

# 精确统计
exact_zeros = (alpha_values == 0).sum()
le_zeros = (alpha_values <= 0).sum()
pos_alphas = (alpha_values > 0).sum()

zero_pct = exact_zeros / len(df) * 100
print(f"\nalpha == 0: {exact_zeros:,} 个 ({zero_pct:.2f}%)")
print(f"alpha <= 0: {le_zeros:,} 个 ({le_zeros/len(df)*100:.2f}%)")
print(f"alpha >  0: {pos_alphas:,} 个 ({pos_alphas/len(df)*100:.2f}%)")

print(f"\nalpha分布:")
print(alpha_values.describe().round(4))

if exact_zeros > 0:
    print(f"\n🚨 alpha=0的csemCur (前10):")
    zero_csem = df[df['alpha'] == 0][['csemCur', 'alpha']].head(10)
    print(zero_csem)
    print(f"\n0值csemCur列表 (前20): {df[df['alpha']==0]['csemCur'].head(20).tolist()}")

if le_zeros > exact_zeros:
    print(f"\n⚠️ 负值alpha (前5):")
    print(df[df['alpha'] < 0][['csemCur', 'alpha']].head())

# 保存
# df[['csemCur', 'alpha']].to_csv('alpha_检测报告.csv', index=False)
print(f"\n💾 保存: alpha_检测报告.csv")

print("\n🎯 建议:")
if zero_pct > 50:
    print("❌ 零值>50%，模拟结果大部分为0！")
elif zero_pct > 20:
    print("⚠️ 零值较多，考虑过滤alpha>0")
else:
    print("✅ 零值比例正常，可直接模拟")

🔍 alpha列零值检测报告 (每个csemCur一行)
总csemCur数量: 224

alpha == 0: 0 个 (0.00%)
alpha <= 0: 0 个 (0.00%)
alpha >  0: 224 个 (100.00%)

alpha分布:
count   224.0000
mean      0.0004
std       0.0005
min       0.0000
25%       0.0001
50%       0.0002
75%       0.0005
max       0.0031
Name: alpha, dtype: float64

💾 保存: alpha_检测报告.csv

🎯 建议:
✅ 零值比例正常，可直接模拟


In [ ]:
### new way of simulating, Ncr(k) = h(k-1)

# 修改后的代码一：成本敏感性分析版本 (替换优化为你的模拟)
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
pd.set_option('display.float_format', '{:.6f}'.format)

# Gamma parameters
gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 0. Read data (使用你的原始逻辑，只取cr>0的部分)
# =============================================================================
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv')
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

# =============================================================================
# 参数 (C_MANT, C_COPER 固定, C_leak 变化)
# =============================================================================
T, base_year, Y = 27, 2024, 365
mu_coper, mu_mant = 1.6, 4.2
C_MANT = 48000
C_COPER = 15600
nb_coper, nb_mant = 3, 14

# =============================================================================
# 1. 2023 baseline
# =============================================================================
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]
print("2023 baseline (所有csemCur):")
print(df_alpha_2023.round(4))

# =============================================================================
# 2. 模拟函数 (你的原始模拟逻辑，返回详细结果)
# =============================================================================
def run_simulation(C_leak):
    # Initialize states (你的原始逻辑)
    csem_states = {}
    for i, row in df_alpha_2023.iterrows():
        csemCur = int(row["csemCur"])
        alpha, delta_p = row["alpha_2023"], row["Pression Nominale Relative"] - row["CR_threshold"]
        cr_init = row["cr"] if row["cr"] > 0 else 0
        
        csem_states[csemCur] = {
            'cr_current': cr_init,
            'age_2023_fixed': Age_2023_fixed[i],
            'current_age': Age_2023_fixed[i] + 1,  # 2024开始
            'alpha': alpha, 'pression_nominale': row["Pression Nominale Relative"],
            'cr_threshold': row["CR_threshold"], 'volume_csem': row["Volume"],
            'csem_K': 146 * row["Volume"] / (8.314 * 293.15 * 10**3)
        }

    # Simulation loop (你的原始逻辑)
    rows = []
    for t in range(T):
        year = base_year + t
        year_offset = year - 2023
        
        # Find triggered & assign tasks (你的逻辑)
        triggered_csem = []
        for csemCur, state in csem_states.items():
            cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
            leak = cr * delta_p * (10**5) * state['csem_K']
            interval = 365 / cr if cr > 0 else np.inf
            if leak >= 5 or interval <= 60:
                triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'state': state})
        
        triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
        task_assignments = {}
        for i, info in enumerate(triggered_csem):
            csemCur = info['csemCur']
            if i < nb_mant: task_assignments[csemCur] = 'mant'
            elif i < nb_mant + nb_coper: task_assignments[csemCur] = 'coper'
        
        # Process all
        for csemCur, state in csem_states.items():
            cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
            leak = cr * delta_p * (10**5) * state['csem_K']
            interval = 365 / cr if cr > 0 else np.inf
            triggered = (leak >= 5) or (interval <= 60)
            
            leak_cost = leak * C_leak  # 使用传入的C_leak
            task_type = task_assignments.get(csemCur)
            action_cost = { 'mant': C_MANT, 'coper': C_COPER }.get(task_type, 0)
            
            rows.append(dict(csemCur=csemCur, year=year, cr=cr, leak=leak, interval=interval,
                            triggered=triggered, task_type=task_type,
                            leak_cost=leak_cost, action_cost=action_cost, total_cost=leak_cost+action_cost))
            
            # Update state (你的逻辑)
            next_offset = year_offset + 1
            next_age = state['age_2023_fixed'] + next_offset
            
            if task_type == 'mant':
                state['cr_current'] /= mu_mant
            elif task_type == 'coper':
                state['cr_current'] /= mu_coper
            else:
                state['cr_current'] = compute_gamma_h(next_age-1, gamma_alpha, gamma_beta)
            state['current_age'] = next_age

    sim_df = pd.DataFrame(rows)
    sim_df['csemCur'] = sim_df['csemCur'].astype(int)
    
    # Summary (你的原始汇总逻辑)
    cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
    cost_summary.columns = ["csemCur", "total_T years_cost"]
    leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
    leak_summary.columns = ["csemCur", "total_leak_kg"]
    
    total_leak_kg = sim_df['leak'].sum().round(3)
    total_cost = sim_df['total_cost'].sum().round(0)
    total_leak_cost = sim_df['leak_cost'].sum().round(0)
    total_action_cost = sim_df['action_cost'].sum().round(0)
    total_coper_actions = (sim_df['task_type'] == 'coper').sum()
    total_mant_actions = (sim_df['task_type'] == 'mant').sum()
    
    print(f"\n✅ C_leak = {C_leak:,.0f} | Total Cost: {total_cost:,.0f}")
    print(f"Total leak: {total_leak_kg:.1f} kg | Action cost: {total_action_cost:,.0f} | Leak cost: {total_leak_cost:,.0f}")
    print(f"Total COPER: {total_coper_actions} | Total MANT: {total_mant_actions}")
    
    return {
        "C_leak": C_leak,
        "total_cost": total_cost,
        "total_leak_kg": total_leak_kg,
        "leak_cost": total_leak_cost,
        "action_cost": total_action_cost,
        "total_coper_actions": total_coper_actions,
        "total_mant_actions": total_mant_actions,
        "status": "simulation_complete"
    }, sim_df

# =============================================================================
# 3. 成本敏感性分析 (模仿优化代码的格式)
# =============================================================================
base_C_leak = 25200 * 0.100  # 基准值
C_leak_list = [
    base_C_leak * 0.01,
    base_C_leak * 0.1,
    base_C_leak * 0.5,
    base_C_leak * 1,
    base_C_leak * 1.5,
    base_C_leak * 2.0,
    base_C_leak * 3.0,
    base_C_leak * 4.0,
    base_C_leak * 5.0,
    base_C_leak * 10.0,
    base_C_leak * 100.0,
    base_C_leak * 1000.0,
    base_C_leak * 10000.0,
    base_C_leak * 100000.0
]

results = []
detailed_dfs = {}  # 保存详细轨迹

for C_leak in C_leak_list:
    res, sim_df = run_simulation(C_leak)
    results.append(res)
    detailed_dfs[C_leak] = sim_df  # 可选：保存详细数据

# Summary table
result_df = pd.DataFrame(results)
print("\n========== Summary of Sensitivity Analysis (Simulation) ==========")
print(result_df.round(0))

# 保存
filename = "simulation_sensitivity_C_leak"
result_df.to_csv(f"{filename}_summary.csv", index=False)
print(f"\n✅ Saved {filename}_summary.csv")

# 统计输出 (模仿优化代码)
print('\nNumber of COPER actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_coper_actions']:.0f} COPER actions") 

print('\nNumber of MANT actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_mant_actions']:.0f} MANT actions")


2023 baseline (所有csemCur):
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold  \
0    5000252447    0.000300                    3.500000      3.199000   
1    5000252448    0.000500                    3.500000      3.199000   
2    5000252449    0.000100                    3.500000      3.199000   
3    5000252450    0.000900                    3.500000      3.199000   
4    5000252524    0.000400                    3.500000      3.199000   
..          ...         ...                         ...           ...   
219  5001342228    0.001000                    3.500000      3.199000   
220  5001342229    0.000300                    3.500000      3.199000   
221  5002343705    0.000300                    3.500000      3.199000   
222  5002343706    0.000100                    3.500000      3.199000   
223  5002343707    0.000200                    3.500000      3.199000   

      Volume       cr  
0   6.317600 0.000000  
1   6.317600 0.000000  
2   6.302500 0.000000  
